In [ ]:
pip list

In [ ]:
# ==================================================
# 1. IMPORTS
# ==================================================
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import scorecardpy as sc
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, 
    accuracy_score, precision_score, recall_score, f1_score
)

# ==================================================
# 2. PATH DINÂMICO (Ajustado para subpastas)
# ==================================================
# Como seu notebook está em /Notebooks/script_pipeline/, subimos dois níveis
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
MODEL_PATH = os.path.join(BASE_DIR, "models")
DATA_PATH = os.path.join(BASE_DIR, "data", "german_credit_data.csv")

os.makedirs(MODEL_PATH, exist_ok=True)

# ==================================================
# 3. LOAD DATA
# ==================================================
df = pd.read_csv(DATA_PATH)

# ==================================================
# 4. FEATURE ENGINEERING
# ==================================================
df1 = df.copy()
df1['Saving accounts'] = df1['Saving accounts'].fillna('little')
df1['Checking account'] = df1['Checking account'].fillna('little')
df1['Saving accounts'] = df1['Saving accounts'].replace('quite rich', 'rich')

df1.rename(columns={
    'Age': 'Idade', 'Sex': 'Genero', 'Job': 'Trabalho',
    'Housing': 'Habitacao', 'Saving accounts': 'Conta_poupanca',
    'Checking account': 'Conta_corrente', 'Credit amount': 'Valor_credito',
    'Duration': 'Duracao', 'Purpose': 'Finalidade', 'Risk': 'Risco'
}, inplace=True)

# ==================================================
# 5. BINNING
# ==================================================
df1["Faixa_Etaria"] = pd.cut(df1["Idade"], [0,28,38,48,58,68,78]).astype(str)
df1["Faixa_Duracao"] = pd.cut(df1["Duracao"], [0,15,27,39,51,63,75]).astype(str)
df1["Faixa_Credito"] = pd.cut(df1["Valor_credito"], range(250,21000,1000)).astype(str)

# ==================================================
# 6. TARGET
# ==================================================
df1["Risco"] = df1["Risco"].map({"good":1, "bad":0})

# ==================================================
# 7. WOE
# ==================================================
df_woe = df1.copy()
df_woe = df_woe.drop(columns=["Idade", "Duracao", "Valor_credito", "Unnamed: 0"], errors="ignore")

for col in ["Faixa_Etaria", "Faixa_Duracao", "Faixa_Credito"]:
    df_woe[col] = df_woe[col].astype(str)

bins = sc.woebin(df_woe, y="Risco")
joblib.dump(bins, os.path.join(MODEL_PATH, "woe_bins.pkl"))

df_woe = sc.woebin_ply(df_woe, bins)
feature_names = df_woe.drop("Risco", axis=1).columns.tolist()

# ==================================================
# 8. SPLIT
# ==================================================
X = df_woe.drop("Risco", axis=1)
y = df_woe["Risco"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# ==================================================
# 9. MODELOS
# ==================================================
models = {
    "Logistic": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, eval_metric="logloss", random_state=42)
}

# ==================================================
# 10. TREINAMENTO
# ==================================================
results=[]
trained_models={}

for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:,1]
    pred = (prob>=0.5).astype(int)
    
    auc = roc_auc_score(y_test, prob)
    ks = np.max(roc_curve(y_test, prob)[1] - roc_curve(y_test, prob)[0])
    
    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "AUC": auc,
        "Gini": 2*auc-1,
        "KS": ks
    })
    trained_models[name] = model

# ==================================================
# 11. MELHOR MODELO
# ==================================================
results_df = pd.DataFrame(results).sort_values(by="KS", ascending=False)
best_model_name = results_df.iloc[0]["Modelo"]
best_model = trained_models[best_model_name]

# ==================================================
# 12. PREVISÃO E MÉTRICAS FINAIS
# ==================================================
y_prob = best_model.predict_proba(X_test)[:,1]
y_prob_train = best_model.predict_proba(X_train)[:,1] # Para o PSI
y_pred = (y_prob>=0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

# ==================================================
# 13. CÁLCULO DO PSI (ESTABILIDADE)
# ==================================================
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.arange(0, buckets + 1) / buckets * 100
    breakpoints = np.percentile(expected, breakpoints)
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    breakpoints = np.unique(breakpoints) # Evita bins duplicados
    
    expected_percents = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    
    expected_percents = np.clip(expected_percents, 0.0001, 1)
    actual_percents = np.clip(actual_percents, 0.0001, 1)
    
    return np.sum((expected_percents - actual_percents) * np.log(expected_percents / actual_percents))

psi_value = calculate_psi(y_prob_train, y_prob)

# ==================================================
# 14. SALVAR ARQUIVOS (MODELOS E CSVs)
# ==================================================
joblib.dump(best_model, os.path.join(MODEL_PATH, "modelo.pkl"))
joblib.dump(feature_names, os.path.join(MODEL_PATH, "feature_names.pkl"))
results_df.to_csv(os.path.join(MODEL_PATH, "comparacao_modelos.csv"), index=False)

# ==================================================
# 15. MÉTRICAS JSON (UNIFICADO COM PSI)
# ==================================================
metricas = {
    "best_model": best_model_name,
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "auc": float(roc_auc_score(y_test, y_prob)),
    "gini": float(2*roc_auc_score(y_test, y_prob)-1),
    "ks": float(results_df.iloc[0]["KS"]),
    "psi": float(psi_value),
    "confusion_matrix": {
        "TN": int(cm[0][0]), "FP": int(cm[0][1]),
        "FN": int(cm[1][0]), "TP": int(cm[1][1])
    }
}

with open(os.path.join(MODEL_PATH, "metricas.json"), "w") as f:
    json.dump(metricas, f, indent=4)

# ==================================================
# 16. PARÂMETROS PDO E CUTOFFS
# ==================================================
score_params = {"pdo": 20, "base_score": 600, "base_odds": 50}
with open(os.path.join(MODEL_PATH, "score_params.json"), "w") as f:
    json.dump(score_params, f, indent=4)

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
ks_idx = np.argmax(tpr - fpr)
best_prob_cutoff = thresholds[ks_idx]

# Score do Cutoff
factor = score_params["pdo"] / np.log(2)
offset = score_params["base_score"] - factor * np.log(score_params["base_odds"])
odds_cutoff = best_prob_cutoff / (1 - best_prob_cutoff)
ks_score_cutoff = int(offset + factor * np.log(odds_cutoff))

ks_cutoffs = {
    "best_prob_cutoff": float(best_prob_cutoff),
    "ks_score_cutoff": ks_score_cutoff,
    "reject_cutoff": ks_score_cutoff - 30,
    "approve_cutoff": ks_score_cutoff + 30
}

with open(os.path.join(MODEL_PATH, "ks_cutoffs.json"), "w") as f:
    json.dump(ks_cutoffs, f, indent=4)

print(f"\n✅ PIPELINE FINALIZADO COM SUCESSO!")
print(f"Modelo: {best_model_name} | KS: {metricas['ks']:.4f} | PSI: {psi_value:.4f}")
print(f"Arquivos salvos em: {MODEL_PATH}")

[INFO] creating woe binning ...
[INFO] converting into woe values ...
KS Cutoffs salvos ✔

PIPELINE FINALIZADO
Modelo vencedor: Logistic
Métricas salvas ✔
Parâmetros PDO salvos ✔
KS Cutoffs salvos ✔

------ POLÍTICA KS ------
Probabilidade cutoff KS: 0.5304
Score cutoff KS: 490
Reject cutoff: 460
Review cutoff: 490
Approve cutoff: 520
